# 03 · Nested

Three levels deep, and no plan. `diagnose` asks `hypothesize` for a theory;
`hypothesize` opens one artifact at a time through `inspect_artifact`, and each
reading decides what to open next.

> **You'll learn**
> - Nest reasoners three deep and see the depth in the graph
> - Let the model choose its own next step instead of following a fixed plan
> - Watch the same incident produce different answers on different runs

In [1]:
import os, sys, json, time, pathlib, httpx
sys.path.insert(0, str(pathlib.Path.cwd().parent / "lib"))
import dag

SERVER = os.environ.get("AGENTFIELD_SERVER", "http://localhost:8080")
GT = {g["id"]: g for g in json.load(open("../incidents/ground_truth.json"))["incidents"]}

def run(reasoner, **inp):
    """Dispatch to the node over HTTP and wait. Returns (run_id, output).

    Async on purpose: it hands back the run_id the DAG needs, and it does not
    die at the control plane's 90s synchronous ceiling.
    (Never `await app.call(...)` from a notebook — it executes children twice.)
    """
    r = httpx.post(f"{SERVER}/api/v1/execute/async/blast-radius.{reasoner}",
                   json={"input": inp}, timeout=30).json()
    eid, rid = r["execution_id"], r["run_id"]
    for _ in range(200):
        time.sleep(3)
        d = httpx.get(f"{SERVER}/api/v1/executions/{eid}", timeout=20).json()
        d = d.get("data", d)
        if d.get("status") in ("succeeded", "completed"):
            return rid, d.get("output") or d.get("result")
        if d.get("status") in ("failed", "error"):
            raise RuntimeError(d.get("error"))
    raise TimeoutError(eid)

def show(dx, title=""):
    if title:
        print(title); print("=" * len(title))
    print("root cause  :", dx["root_cause"])
    print("remediation :", dx["remediation"])
    print("confident   :", dx["confident"])
    for f in dx["findings"]:
        print(f"  - [{f['severity']:<8}] {f['location']}: {f['claim']}")


The trail is not written down anywhere. `inspect_artifact` returns a
`next_artifact` field, and `hypothesize` follows it.

In [2]:
print(open("../node/rungs/r03.py").read())

"""r03 — nested, improvised.

`diagnose` asks for a hypothesis. `hypothesize` decides for itself which artifact to
open next and calls `inspect_artifact` to read it. Nothing about the path is fixed,
so the graph is different every run — and so, often, is the answer.
"""
from agentfield import AgentRouter
from pydantic import BaseModel, Field

from common import MODEL, NODE_ID, SYSTEM, Diagnosis, incident_text

router = AgentRouter(prefix="r03", tags=["rung", "r03"])

ARTIFACTS = ["alert", "logs", "deploys", "metrics", "topology"]
MAX_LOOKS = 3


class Reading(BaseModel):
    """What one artifact turned out to say."""

    artifact: str
    what_it_shows: str = Field(description="Two or three sentences. Only what this artifact supports.")
    quote: str = Field(description="A literal line copied from the artifact")
    next_artifact: str = Field(description=f"One of {ARTIFACTS}, or 'none' if the picture is complete")


class Hypothesis(BaseModel):
    """A theory, and the trail that pro

## 1 · The same incident, three times

Nothing changes between these three runs. Same incident, same node, same model.

In [3]:
import concurrent.futures as cf

with cf.ThreadPoolExecutor(3) as ex:
    futures = [ex.submit(run, "r03_diagnose", incident_id="inc-008") for _ in range(3)]
    runs = [f.result() for f in futures]

for i, (rid, d) in enumerate(runs, 1):
    print(f"--- run {i}  ({rid})")
    print("  root cause :", d["root_cause"])
    print("  remediation:", d["remediation"])
    print("  confident  :", d["confident"])
    print()

--- run 1  (run_20260820_124035_jrwdmw26)
  root cause : Notification-worker OOMKilled due to a listener leak in TemplateRenderer.ts where each render call registers a listener on the module-level metrics emitter, causing heap exhaustion from 1.8 million retained EventEmitter[render] instances.
  remediation: Fix the render metrics hook in TemplateRenderer.ts to avoid registering a new listener per render call; instead, use a single persistent listener or batch updates to the metrics emitter.
  confident  : True

--- run 2  (run_20260820_124035_r3xgkg72)
  root cause : The deploy dep-2201 introduced a metrics hook that registers a listener per render call on the shared module-level EventEmitter without removal, causing over 1.8 million retained listeners consuming 812 MB of heap, resulting in repeated OOM kills.
  remediation: Remove listeners after each render call, or increase the memory limit temporarily while the listener leak is fixed.
  confident  : True

--- run 3  (run_20260820

Read the remediation column, not just the root cause. That is where the three runs
part company.

In [4]:
from IPython.display import Markdown

rows = "\n".join(
    f"| {i} | {d['root_cause']} | {d['remediation']} |"
    for i, (_, d) in enumerate(runs, 1)
)
Markdown("| run | root cause | remediation |\n|---|---|---|\n" + rows)

| run | root cause | remediation |
|---|---|---|
| 1 | Notification-worker OOMKilled due to a listener leak in TemplateRenderer.ts where each render call registers a listener on the module-level metrics emitter, causing heap exhaustion from 1.8 million retained EventEmitter[render] instances. | Fix the render metrics hook in TemplateRenderer.ts to avoid registering a new listener per render call; instead, use a single persistent listener or batch updates to the metrics emitter. |
| 2 | The deploy dep-2201 introduced a metrics hook that registers a listener per render call on the shared module-level EventEmitter without removal, causing over 1.8 million retained listeners consuming 812 MB of heap, resulting in repeated OOM kills. | Remove listeners after each render call, or increase the memory limit temporarily while the listener leak is fixed. |
| 3 | The metrics hook added in deploy dep-2201 registers a listener on the shared metrics emitter for each render call, but never removes them, causing unbounded EventEmitter listener accumulation that exhausts heap memory and triggers OOMKilled restarts. | Immediately remove the metrics hook or modify it to reuse a single listener and ensure listeners are removed after use; as a temporary stopgap, increase the memory limit beyond 1.5 Gi to buy time. |

Ground truth, so you can score them yourself.

In [5]:
g = GT["inc-008"]
print("actual root cause:\n ", g["root_cause"]["summary"], "\n")
print("must include:", g["root_cause_must_include"], "\n")
for i, (_, d) in enumerate(runs, 1):
    hit = [k for k in g["root_cause_must_include"] if k.lower() in d["root_cause"].lower()]
    print(f"run {i}: matched {hit or 'nothing'}")

actual root cause:
  Release 5.2.0 (dep-2201, nine days earlier) added a metrics hook that attaches a listener to a module-level EventEmitter on every render and never removes it. Each listener retains its RecipientContext, so RSS grows monotonically with cumulative messages processed rather than with concurrency. Daily peak RSS climbed from ~405MB to 1394MB over nine days and crossed the 1.5Gi limit during this morning's ordinary peak, producing the OOM kill loop. 

must include: ['leak', 'listener', '5.2.0'] 

run 1: matched ['leak', 'listener']
run 2: matched ['listener']
run 3: matched ['listener']


In [6]:
for i, (_, d) in enumerate(runs, 1):
    bad = [w for w in g["wrong_remediations"] if w.split()[0].lower() in d["remediation"].lower()]
    print(f"run {i}: remediation contains a known-wrong action ->", bad or "no")

run 1: remediation contains a known-wrong action -> no
run 2: remediation contains a known-wrong action -> no
run 3: remediation contains a known-wrong action -> no


Same mechanism, different advice. The keyword check is crude on purpose — the divergence that matters is in what each run would have you do first.

## 2 · The graphs differ too

The answer varies because the *path* varies. Each run opened a different set of
artifacts in a different order, so each run has a different shape.

In [7]:
dag.render_two(runs[0][0], runs[1][0], labels=("r03 · run 1", "r03 · run 2"))

```mermaid
flowchart LR
  subgraph ag["r03 · run 1 — 5 exec · depth 3 · fan-out 3"]
  direction TD
    a0["r03_diagnose<br/><small>✓ succeeded · 36.8s</small>"]
    a1["r03_hypothesize<br/><small>✓ succeeded · 27.7s</small>"]
    a2["r03_inspect_artifact<br/><small>✓ succeeded · 5.5s</small>"]
    a3["r03_inspect_artifact<br/><small>✓ succeeded · 6.5s</small>"]
    a4["r03_inspect_artifact<br/><small>✓ succeeded · 11.9s</small>"]
    a0 --> a1
    a1 --> a2
    a1 --> a3
    a1 --> a4
    class a0,a1,a2,a3,a4 ok;
  end
  subgraph bg["r03 · run 2 — 5 exec · depth 3 · fan-out 3"]
  direction TD
    b0["r03_diagnose<br/><small>✓ succeeded · 60.1s</small>"]
    b1["r03_hypothesize<br/><small>✓ succeeded · 25.5s</small>"]
    b2["r03_inspect_artifact<br/><small>✓ succeeded · 5.5s</small>"]
    b3["r03_inspect_artifact<br/><small>✓ succeeded · 4.6s</small>"]
    b4["r03_inspect_artifact<br/><small>✓ succeeded · 11.6s</small>"]
    b0 --> b1
    b1 --> b2
    b1 --> b3
    b1 --> b4
    class b0,b1,b2,b3,b4 ok;
  end
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

In [8]:
dag.render(runs[2][0], title="r03 · run 3")

**r03 · run 3** — 5 executions · depth 3 · max fan-out 3 · 1 agent(s)

```mermaid
flowchart TD
  n0["r03_diagnose<br/><small>✓ succeeded · 29.7s</small>"]
  n1["r03_hypothesize<br/><small>✓ succeeded · 19.9s</small>"]
  n2["r03_inspect_artifact<br/><small>✓ succeeded · 3.6s</small>"]
  n3["r03_inspect_artifact<br/><small>✓ succeeded · 6.8s</small>"]
  n4["r03_inspect_artifact<br/><small>✓ succeeded · 5.4s</small>"]
  n0 --> n1
  n1 --> n2
  n1 --> n3
  n1 --> n4
  class n0,n1,n2,n3,n4 ok;
  classDef ok   fill:#dcfce7,stroke:#16a34a,stroke-width:1px,color:#14532d;
  classDef run  fill:#dbeafe,stroke:#2563eb,stroke-width:1px,color:#1e3a8a;
  classDef wait fill:#f1f5f9,stroke:#94a3b8,stroke-width:1px,color:#334155;
  classDef bad  fill:#fee2e2,stroke:#dc2626,stroke-width:1px,color:#7f1d1d;
```

This is the blast radius. The agent is not simply wrong; it is differently wrong each
time you ask, along a path it chose at run time — and no single run tells you which
path you got.

## What you learned

- Nesting reasoners adds depth to the graph, one level per `call`.
- An improvised plan is a run-time artifact, not source code — the trail differs each run.
- Three runs of one incident, three different answers. Variance is the thing you have to
  measure.

**Next:** `04_planned` — pin the plan down, and get the same graph every time.